<a href="https://colab.research.google.com/github/nitinog10/Mini-SLM/blob/main/Minislm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Install all required libraries
!pip install torch transformers datasets tokenizers numpy tqdm

In [ ]:
# STEP 2 - LOAD REAL DATASETS (combine all three)
from datasets import load_dataset
import os

all_texts = []

try:
    print("Loading Wikipedia (via wikimedia/wikipedia)...")
    # Dataset 1: Wikipedia (English) - Using the 2023 snapshot available on the Hub
    wiki = load_dataset("wikimedia/wikipedia", "20231101.en", split="train[:5%]")
    all_texts.extend(wiki["text"])
    print(f"Wiki loaded: {len(wiki)} documents")

    print("Loading BookCorpus...")
    # Dataset 2: BookCorpus
    books = load_dataset("bookcorpus", split="train[:5%]")
    all_texts.extend(books["text"])
    print(f"Books loaded: {len(books)} documents")

    print("Loading OpenWebText...")
    # Dataset 3: OpenWebText
    web = load_dataset("Skylion007/openwebtext", split="train[:3%]")
    all_texts.extend(web["text"])
    print(f"Web loaded: {len(web)} documents")

except Exception as e:
    print(f"An error occurred while loading datasets: {e}")

# Combine and report stats
total_docs = len(all_texts)
total_chars = sum(len(t) for t in all_texts)

print(f"\n--- Dataset Summary ---")
print(f"Total documents: {total_docs:,}")
print(f"Total character count: {total_chars:,}")

In [ ]:
# STEP 3 - TRAIN A REAL BPE TOKENIZER
from tokenizers import ByteLevelBPETokenizer
import os

# Initialize the tokenizer
tokenizer = ByteLevelBPETokenizer()

# Create a directory for the tokenizer
os.makedirs("my_tokenizer", exist_ok=True)

# Train the tokenizer
tokenizer.train_from_iterator(
    all_texts,
    vocab_size=16000,
    min_frequency=2,
    special_tokens=["<pad>", "<unk>", "<bos>", "<eos>"]
)

# Save the tokenizer
tokenizer.save_model("my_tokenizer")
print(f"Tokenizer trained. Vocabulary size: {tokenizer.get_vocab_size()}")

In [ ]:
# STEP 4 - TOKENIZE AND PREPARE THE DATASET
import torch
import numpy as np
from torch.utils.data import DataLoader, TensorDataset

print("Tokenizing all texts...")
# Tokenize and flatten into one large list of IDs
full_token_ids = []
for text in all_texts:
    enc = tokenizer.encode(text)
    full_token_ids.extend(enc.ids)

# Convert to numpy for faster processing
full_token_ids = np.array(full_token_ids)

# Create sliding window chunks
max_len = 512
stride = 256
chunks = []

for i in range(0, len(full_token_ids) - max_len, stride):
    chunks.append(full_token_ids[i : i + max_len])

# Convert to torch tensors
tokens_tensor = torch.tensor(np.array(chunks), dtype=torch.long)
print(f"Created {tokens_tensor.shape[0]} chunks of length {max_len}.")

# Split into 90% train and 10% validation
train_size = int(0.9 * len(tokens_tensor))
val_size = len(tokens_tensor) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(tokens_tensor, [train_size, val_size])

print(f"Train dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(val_dataset)}")

In [ ]:
# 2. Download and prepare the wikitext-2-raw-v1 dataset
from datasets import load_dataset

dataset = load_dataset('wikitext', 'wikitext-2-raw-v1')
train_texts = [x['text'] for x in dataset['train'] if len(x['text']) > 0]
val_texts = [x['text'] for x in dataset['validation'] if len(x['text']) > 0]

print(f"Loaded {len(train_texts)} training samples.")

In [ ]:
# 3. Train a BPE tokenizer
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
trainer = BpeTrainer(vocab_size=8000, special_tokens=["[UNK]", "[CLS]", "[SEP]", "[PAD]", "[MASK]"])
tokenizer.pre_tokenizer = Whitespace()

# Train the tokenizer on the training text
tokenizer.train_from_iterator(train_texts, trainer)
tokenizer.save("tokenizer.json")
print("Tokenizer trained and saved.")

In [ ]:
# 4. Build a GPT-style Transformer model in PyTorch
import torch
import torch.nn as nn
import math

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class SLMConfig:
    vocab_size = 8000
    emb_dim = 256
    n_layers = 4
    n_heads = 4
    ff_dim = 1024
    max_len = 256
    dropout = 0.1

class MultiHeadAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n_heads = cfg.n_heads
        self.head_dim = cfg.emb_dim // cfg.n_heads
        self.wq = nn.Linear(cfg.emb_dim, cfg.emb_dim)
        self.wk = nn.Linear(cfg.emb_dim, cfg.emb_dim)
        self.wv = nn.Linear(cfg.emb_dim, cfg.emb_dim)
        self.wo = nn.Linear(cfg.emb_dim, cfg.emb_dim)
        self.dropout = nn.Dropout(cfg.dropout)

    def forward(self, x, mask=None):
        b, t, c = x.size()
        q = self.wq(x).view(b, t, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.wk(x).view(b, t, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.wv(x).view(b, t, self.n_heads, self.head_dim).transpose(1, 2)

        attn = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        if mask is not None:
            attn = attn.masked_fill(mask == 0, float('-inf'))
        attn = torch.softmax(attn, dim=-1)
        attn = self.dropout(attn)

        out = (attn @ v).transpose(1, 2).contiguous().view(b, t, c)
        return self.wo(out)

class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.emb_dim)
        self.attn = MultiHeadAttention(cfg)
        self.ln2 = nn.LayerNorm(cfg.emb_dim)
        self.ff = nn.Sequential(
            nn.Linear(cfg.emb_dim, cfg.ff_dim),
            nn.GELU(),
            nn.Linear(cfg.ff_dim, cfg.emb_dim),
            nn.Dropout(cfg.dropout)
        )

    def forward(self, x, mask=None):
        x = x + self.attn(self.ln1(x), mask)
        x = x + self.ff(self.ln2(x))
        return x

class SLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.emb_dim)
        self.pos_emb = nn.Parameter(torch.zeros(1, cfg.max_len, cfg.emb_dim))
        self.blocks = nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg.n_layers)])
        self.ln_f = nn.LayerNorm(cfg.emb_dim)
        self.head = nn.Linear(cfg.emb_dim, cfg.vocab_size, bias=False)

    def forward(self, idx, targets=None):
        b, t = idx.size()
        mask = torch.tril(torch.ones(t, t)).to(device)
        x = self.tok_emb(idx) + self.pos_emb[:, :t, :]
        for block in self.blocks:
            x = block(x, mask)
        logits = self.head(self.ln_f(x))

        loss = None
        if targets is not None:
            loss = nn.functional.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

cfg = SLMConfig()
model = SLM(cfg).to(device)
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# 5. Efficient DataLoader
import numpy as np

def get_batches(data, batch_size, seq_len):
    # Flatten data and chunk it
    n_batches = len(data) // (batch_size * seq_len)
    data = data[:n_batches * batch_size * seq_len]
    data = data.reshape(batch_size, -1)
    for i in range(0, data.shape[1] - seq_len, seq_len):
        x = data[:, i:i+seq_len]
        y = data[:, i+1:i+seq_len+1]
        yield torch.tensor(x, dtype=torch.long).to(device), torch.tensor(y, dtype=torch.long).to(device)

# Tokenize full dataset once
all_tokens = []
for text in train_texts:
    all_tokens.extend(tokenizer.encode(text).ids)
train_data = np.array(all_tokens)

In [ ]:
# 6. Full Training Loop
from torch.optim import AdamW
from tqdm import tqdm

optimizer = AdamW(model.parameters(), lr=3e-4)
model.train()

epochs = 3
step = 0
for epoch in range(epochs):
    batch_gen = get_batches(train_data, batch_size=32, seq_len=cfg.max_len)
    for x, y in batch_gen:
        logits, loss = model(x, y)

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        if step % 100 == 0:
            print(f"Epoch {epoch+1} | Step {step} | Loss: {loss.item():.4f}")
        step += 1

In [ ]:
# 7. Text Generation Function
@torch.no_grad()
def generate(model, tokenizer, prompt, max_new_tokens=100, k=10):
    model.eval()
    ids = torch.tensor(tokenizer.encode(prompt).ids, dtype=torch.long).unsqueeze(0).to(device)

    for _ in range(max_new_tokens):
        # Crop context if needed
        idx_cond = ids[:, -cfg.max_len:]
        logits, _ = model(idx_cond)
        logits = logits[:, -1, :] # Get last token logits

        # Top-k sampling
        v, _ = torch.topk(logits, k)
        logits[logits < v[:, [-1]]] = -float('Inf')
        probs = torch.softmax(logits, dim=-1)

        next_id = torch.multinomial(probs, num_samples=1)
        ids = torch.cat((ids, next_id), dim=1)

    return tokenizer.decode(ids[0].tolist())

In [ ]:
# 8. Test the model
prompt = "The history of science"
generated_text = generate(model, tokenizer, prompt)
print(f"Prompt: {prompt}\nGenerated: {generated_text}")

In [ ]:
# 9. Save and Download model
torch.save(model.state_dict(), 'slm_model.pt')
from google.colab import files
# files.download('slm_model.pt') # Uncomment to trigger browser download

In [ ]:
# 10. Generate text with a different prompt
new_prompt = "In the distant future, artificial intelligence"
new_generated_text = generate(model, tokenizer, new_prompt, max_new_tokens=100)
print(f"Prompt: {new_prompt}\nGenerated: {new_generated_text}")

In [ ]:
new_prompt = "what is artificial intelligence"
new_generated_text = generate(model, tokenizer, new_prompt, max_new_tokens=100)
print(f"Prompt: {new_prompt}\nGenerated: {new_generated_text}")